# RocketKV Implementation (16x compression) -- two-stage (SnapKV eviction + Hybrid Sparse Attention) -- KVQuant-family harness (GSM8K + ARC-Challenge + HellaSwag)

> **READ THIS FIRST -- what this notebook is and is NOT.**
>
> RocketKV (Behnam et al., ICML 2025, arXiv:2502.14051) is a **long-context, decode-phase** KV-cache
> compression method. Its published results use token budgets of 256-4096 against sequences of
> **16K-109K tokens** (LongBench, Needle-in-a-Haystack, RULER, SCBench), where it reaches compression
> ratios up to 400x. GSM8K / ARC-Challenge / HellaSwag are **short-context** tasks (prompts here are
> roughly 0.8K-3K tokens, generation 8-256 tokens). RocketKV was never designed or evaluated in this
> regime.
>
> **Consequently, the numbers this notebook produces on these three datasets are NOT a valid
> RocketKV result and MUST NOT be placed in a comparison table against H2O / KVQuant as if the
> regimes matched.** Two failure modes are expected and are properties of the regime, not bugs:
>   1. **No-op at high budgets.** Whenever the token budget >= the prompt length, RocketKV evicts
>      nothing and sparsifies nothing, so its output is identical to the full-KV baseline.
>   2. **Out-of-regime at low budgets.** At budgets small enough to engage (e.g. 64/128/256 on a ~1K
>      prompt) the method runs far outside its design point; stage-1's fixed 32-token observation
>      window and stage-2's nested budget splits can degenerate, so accuracy may collapse for reasons
>      that are budget-rounding artifacts rather than informative signal.
>
> **This notebook exists as (a) a functional / smoke-test implementation of RocketKV behind the exact
> same harness interface as the H2O and KVQuant notebooks, and (b) a reusable engine for when
> long-context datasets (LongBench / RULER) are added later**, at which point RocketKV becomes a
> meaningful comparison. Peak memory here is **peak KV-CACHE bytes across BOTH prefill and decode**, measured the SAME
> tensor-byte way as the H2O / KVQuant / baseline notebooks (numel x element_size over K,V across
> layers) -- NOT total GPU memory. RocketKV's high-water mark is the full dense prompt cache at
> end of prefill, before stage-1 eviction (RocketKV must reconstruct the whole prompt cache to
> score what to keep). That dense value is budget-INDEPENDENT, so peak KV-cache is largely the
> SAME across the compression levels; the levels are kept because LATENCY and ACCURACY still
> differ. Note H2O (evicts during prefill) and KVQuant (quantizes at storage) have NO dense
> prefill spike, so on this same metric RocketKV will show a higher peak than they do -- a real
> architectural cost of SnapKV-style scoring, not a measurement artifact.
>
> A VALIDATION cell below checks the one invariant that IS verifiable on short
> data: at a budget >= sequence length, RocketKV must reproduce the full-KV baseline exactly.
>
> **Implementation note.** The official RocketKV code (NVlabs/RocketKV, vendored as a submodule) is
> built on `gpt-fast`, not HuggingFace `transformers`, and ships no HF-compatible attention engine
> (unlike H2O's `H2OKVCache_LayerWise`). Both stages below are therefore a **faithful
> reimplementation** of the paper's algorithm (Section 3, Algorithm 1) as an eager-attention path
> matching this harness -- not a call into the authors' code. It has not been validated against the
> paper's published numbers (which do not exist for this regime); treat the two stages as
> needing a GPU shakeout run before any use beyond the smoke test.

Harness parity with the rest of the family: same model, pinned package versions, seed, dataset
loading / tokenization, GSM8K prompt + question set + answer grading, metric definitions, and
result-CSV schema. Only the compression engine and the per-question generate/evaluate functions
differ. Like H2O, RocketKV needs real attention weights (SnapKV stage 1 scores tokens from them), so
the model is loaded with `attn_implementation="eager"`; latency therefore carries the same "eager
tax" caveat as the H2O notebook, and is not deployment-representative.

Run cells top to bottom. Needs a GPU runtime.

## Setup

In [ ]:
!hostname

In [ ]:
# Block 1 - Environment setup
# Run once per fresh runtime. Package versions are pinned so environment
# differences are never a confound between compression methods -- kept
# byte-for-byte identical to the KVQuant-family notebooks (including
# datasets==2.14.5, which the previous H2O notebook left unpinned).

from google.colab import drive
drive.mount("/content/drive")

!python -m pip install -q --no-deps \
  "transformers==4.43.4" \
  "accelerate==0.33.0" \
  "tokenizers==0.20.3" \
  "huggingface_hub==0.36.2" \
  sentencepiece \
  einops

!python -m pip install -q \
  "datasets==2.14.5" \
  tqdm \
  matplotlib

!python -m pip install -q --no-deps --force-reinstall "huggingface_hub==0.36.2"

import os

# Patch - transformers==4.43.4 hard-enforces "tokenizers>=0.19,<0.20" at
# IMPORT time (transformers/dependency_versions_check.py), not just at
# pip-install time -- so even though tokenizers==0.20.3 installs fine
# (needed since 0.19.x has no Python 3.13 wheel), "import transformers"
# still raises ImportError unless this hardcoded constraint is relaxed on
# disk first. Patched via a direct file edit -- importing transformers to
# patch it in-memory is not an option, since that import is exactly what
# triggers the failing check. Safe to run even on the old (working)
# environment: on Python <3.13 this is a no-op the moment the constraint
# line has already been relaxed, and re-running it is idempotent.
import importlib.util
import re

_transformers_spec = importlib.util.find_spec("transformers")
_deps_table_path = os.path.join(os.path.dirname(_transformers_spec.origin), "dependency_versions_table.py")

with open(_deps_table_path, "r") as f:
    _deps_content = f.read()

_deps_content_new, _n_subs = re.subn(
    r'("tokenizers":\s*)"tokenizers>=0\.19,<0\.20"',
    r'\1"tokenizers>=0.19,<0.21"',
    _deps_content,
)
if _n_subs > 0:
    with open(_deps_table_path, "w") as f:
        f.write(_deps_content_new)
    print(f"Patched {_deps_table_path}: tokenizers constraint relaxed to <0.21 "
          "(allows tokenizers==0.20.3 -- 0.19.x has no Python 3.13 wheel).")
else:
    print(f"NOTE: tokenizers constraint in {_deps_table_path} was not the expected "
          "'>=0.19,<0.20' (already patched, or transformers version differs) -- "
          "no change made.")

try:
    from google.colab import userdata
    _hf_token = userdata.get("HF_TOKEN")
except Exception:
    _hf_token = os.environ.get("HF_TOKEN")

if _hf_token:
    from huggingface_hub import login
    login(token=_hf_token)
    print("Logged in to HuggingFace")
else:
    print("No HF_TOKEN found -- Llama-3.1-8B is GATED: this will fail to load without a token that has accepted the Meta license at https://huggingface.co/meta-llama/Llama-3.1-8B")

print("Block 1 finished. Now run Block 2.")

In [ ]:
# Block 2 - Imports, GPU check

import gc
import math
import os
import re
import shutil
import time
import random
import pickle
import sys

import numpy as np
import torch
import torch.nn as nn
import pandas as pd

import datasets
import transformers
import huggingface_hub
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO CUDA")

HAS_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if HAS_CUDA else "cpu")
MODEL_DTYPE = torch.bfloat16 if HAS_CUDA else torch.float32


def clear_hf_dataset_cache(*dataset_names):
    """Removes cached files for the given HF dataset repo name(s) (e.g.
    "wikitext", "gsm8k") from both the datasets cache and the hub cache.
    Used as an on_retry hook: a download that breaks partway through can
    leave a corrupted partial file that every subsequent retry just
    resumes (and re-breaks at the same point) instead of truly restarting
    -- clearing the cache forces a genuinely fresh download."""
    home = os.path.expanduser("~")
    for name in dataset_names:
        for base in [
            os.path.join(home, ".cache", "huggingface", "datasets", name),
            os.path.join(home, ".cache", "huggingface", "hub", f"datasets--{name}"),
        ]:
            shutil.rmtree(base, ignore_errors=True)


if not HAS_CUDA:
    print("WARNING: No GPU detected. This will be very slow.")

# NOTE: clear_hf_dataset_cache is defined here (not in Helper Functions,
# below) for consistency with the KVQuant-family notebooks, where a
# calibration step needs it available early in Setup. This notebook has no
# such dependency itself, but keeping the split identical across the whole
# notebook family avoids Helper Functions containing different things in
# different notebooks. robust_call/sync_if_cuda/clear_memory have no early
# dependency anywhere and live in Helper Functions with the rest of the
# genuinely cross-dataset machinery.

In [ ]:
# Block 3 - Experiment settings.
# Every SHARED quantity (seed, sample counts, few-shot settings, GSM8K prompt)
# is byte-for-byte identical to the H2O / KVQuant / baseline notebooks, so the
# methods differ only where the compression method itself makes them differ.
# The only method-specific settings are the RocketKV token budgets and the
# fixed SnapKV/HSA hyperparameters from the paper.

LOCAL_MODEL_PATH = "/content/llama-3.1-8b"
HF_MODEL_ID = "meta-llama/Llama-3.1-8B"
MODEL_ID = LOCAL_MODEL_PATH if os.path.exists(LOCAL_MODEL_PATH) else HF_MODEL_ID

SHARED_SEED = 42
QA_EVAL_SAMPLES = 2048
GSM8K_MAX_NEW_TOKENS = 256

# ---- RocketKV method hyperparameters ----------------------------------------
# TOKEN_BUDGETS: the total per-attention-group KV token budget t. RocketKV's
# total compression ratio is c = S / t for a sequence of length S. NOTE (see
# the top-of-notebook caveat): on these short-context datasets, any budget
# >= the prompt length is a NO-OP (reduces to full-KV). The paper's budgets are
# 256..4096 against 16K-109K sequences; here we ALSO include smaller budgets so
# the method actually engages on ~1K-3K prompts. Each budget is run as its own
# METHOD_NAME row, exactly like the 20/40/60 H2O variants are separate notebooks.
# ---- RocketKV budget: PROPORTIONAL (fraction of prompt length) by default ----
# BUDGET_MODE controls how the per-attention-group KV token budget t is chosen
# for a prompt of length S:
#   "proportional": fix the COMPRESSION RATIO c and derive t = ceil(S / c) per
#                   prompt. c stays constant across datasets of different lengths
#                   (short now, long later), so the three levels mean the SAME
#                   thing everywhere and sit on the same footing as H2O's
#                   prompt-fraction budget. This is the EXPERIMENTAL axis, and
#                   (measured as post-eviction decode-phase KV-cache bytes) it is
#                   what makes the three levels separate on memory.
#   "absolute":     fix t directly (256/512/1024). Use ONLY to reproduce /
#                   validate against the paper, whose tables are indexed by
#                   absolute budget. This is the VALIDATION axis.
# S is the PROMPT length (same basis as the H2O budget); set inside prefill.
BUDGET_MODE = "proportional"
COMPRESSION_RATIOS = [16]              # THIS notebook = one level (c = S/t). Raise
                                     # (e.g. 64, 128) to reach RocketKV's high-
                                     # compression regime on long datasets.
TOKEN_BUDGETS = [256, 512, 1024]     # used only when BUDGET_MODE == "absolute"
ROCKETKV_LEVELS = COMPRESSION_RATIOS if BUDGET_MODE == "proportional" else TOKEN_BUDGETS

# Fixed hyperparameters (paper Section 3.3-3.6 / Appendix A.3):
SNAPKV_OBS_WINDOW = 32      # observation window at the end of the prompt (stage 1)
SNAPKV_KERNEL_SIZE = 63     # pooling kernel for stage-1 eviction (RocketKV uses 63, not SnapKV's 7)
HSA_MIN_PAGE_SIZE = 1       # page size floor for stage-2 sequence-dim reduction
ROCKETKV_R_MIN = 0.2        # adaptive split-factor clamp (paper: r in [0.2, 0.8])
ROCKETKV_R_MAX = 0.8

random.seed(SHARED_SEED)
np.random.seed(SHARED_SEED)
torch.manual_seed(SHARED_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SHARED_SEED)

GSM8K_FEWSHOT_PREFIX = (
    "You are solving grade-school math word problems.\n"
    "Show the calculation step by step, then end with exactly this format:\n"
    "#### <final number>\n\n"

    "Question: There are 15 trees in the grove. Grove workers will plant trees today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?\n"
    "Answer: There are 15 trees originally. After planting, there are 21 trees. So the workers planted 21 - 15 = 6 trees.\n"
    "#### 6\n\n"

    "Question: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?\n"
    "Answer: There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5 cars.\n"
    "#### 5\n\n"

    "Question: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?\n"
    "Answer: Leah and her sister started with 32 + 42 = 74 chocolates. After eating 35, they have 74 - 35 = 39 left.\n"
    "#### 39\n\n"

    "Question: Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops did Jason give to Denny?\n"
    "Answer: Jason started with 20 lollipops and now has 12. So he gave away 20 - 12 = 8 lollipops.\n"
    "#### 8\n\n"

    "Question: Shawn has five toys. For Christmas, he got two toys each from his mom and dad. How many toys does he have now?\n"
    "Answer: Shawn started with 5 toys. He got 2 from mom and 2 from dad, which is 2 + 2 = 4 more toys. 5 + 4 = 9 toys total.\n"
    "#### 9\n\n"

    "Question: There were nine computers in the server room. Five more computers were installed each day, from Monday to Thursday. How many computers are now in the server room?\n"
    "Answer: 4 days from Monday to Thursday, with 5 computers installed each day, is 4 * 5 = 20 computers added. 9 + 20 = 29 computers total.\n"
    "#### 29\n\n"

    "Question: Michael had 58 golf balls. On Tuesday, he lost 23 golf balls. On Wednesday, he lost 2 more. How many golf balls did he have at the end of Wednesday?\n"
    "Answer: Michael started with 58 golf balls. After losing 23 on Tuesday, he had 58 - 23 = 35. After losing 2 more on Wednesday, he had 35 - 2 = 33 golf balls.\n"
    "#### 33\n\n"

    "Question: Olivia has $23. She bought five bagels for $3 each. How much money does she have left?\n"
    "Answer: Five bagels at $3 each cost 5 * 3 = 15 dollars. Olivia started with $23, so she has 23 - 15 = 8 dollars left.\n"
    "#### 8\n"
)

# bf16 like the baseline/H2O notebooks (NOT fp16 -- that constraint is KVQuant-only).
MODEL_DTYPE = torch.bfloat16 if HAS_CUDA else torch.float32

print("Model:", MODEL_ID)
print("Random sampling seed:", SHARED_SEED)
print("Eval samples per dataset:", QA_EVAL_SAMPLES)
print("GSM8K max new tokens:", GSM8K_MAX_NEW_TOKENS)
print("RocketKV budget mode:", BUDGET_MODE, "| levels:", ROCKETKV_LEVELS,
      ("(compression ratios c=S/t)" if BUDGET_MODE=="proportional" else "(absolute token budgets)"))
print("SnapKV obs window / kernel:", SNAPKV_OBS_WINDOW, "/", SNAPKV_KERNEL_SIZE)

# NOTE: ARC-Challenge/HellaSwag no longer take few-shot exemplars or a
# capped generation length -- they are scored zero-shot via teacher-forced
# per-choice likelihood (see the Shared multiple-choice (MC) scoring
# machinery cell, below), matching large_sample_implementations exactly.
# GSM8K is unaffected: it keeps GSM8K_FEWSHOT_PREFIX above.


In [ ]:
# Repo setup - Clone the KVCacheCompression repo (always fresh, matching the
# KVQuant notebooks' clean-clone convention) and initialize ONLY the H2O
# submodule -- the official FMInference/H2O repository. The other submodules
# (KVQuant, KIVI, SnapKV, ...) are not needed here, so a targeted init keeps
# this much faster than --recurse-submodules.
#
# From the submodule we import H2OKVCache_LayerWise from
# h2o_hf/utils_real_drop/modify_llama.py: the authors' real-KV-dropping
# eviction engine (per-layer, per-head heavy-hitter scores; physical pruning
# of the cache tensors). The module's other contents (H2OLlamaAttention /
# H2OLlamaForCausalLM) target an older transformers API and MHA-only models,
# so they are not used -- see the notebook intro for details.

if os.path.exists("/content/KVCacheCompression"):
    shutil.rmtree("/content/KVCacheCompression")
    print("Removed existing repo copy for a clean re-clone")

!git clone https://github.com/yoshikodes/KVCacheCompression.git /content/KVCacheCompression
%cd /content/KVCacheCompression
!git submodule update --init --depth 1 H2O

_h2o_module_path = "/content/KVCacheCompression/H2O/h2o_hf/utils_real_drop/modify_llama.py"
assert os.path.exists(_h2o_module_path), \
    "ERROR: official H2O module not found -- clone or submodule init may have failed."

sys.path.insert(0, "/content/KVCacheCompression/H2O/h2o_hf")
from utils_real_drop.modify_llama import H2OKVCache_LayerWise

print("Imported official H2O engine:", H2OKVCache_LayerWise,
      "\nfrom", _h2o_module_path)

In [ ]:
# Block - Load tokenizer + the stock model with EAGER attention. RocketKV's
# stage 1 (SnapKV) scores prompt tokens from real attention weights, which the
# fused SDPA/FlashAttention kernels do not return -- so, exactly like the H2O
# notebook, the model must run eager. This is a method requirement, and it
# carries the same "eager tax" latency caveat: RocketKV latency here is not
# deployment-representative (a real deployment uses the authors' fused kernels).

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

_model_kwargs = {
    "torch_dtype": MODEL_DTYPE,
    "low_cpu_mem_usage": True,
    "attn_implementation": "eager",   # RocketKV/SnapKV need real attention weights
    "trust_remote_code": True,
}
if HAS_CUDA:
    _model_kwargs["device_map"] = {"": 0}

print("Loading stock model (RocketKV modifies the cache at runtime, not the model)...")
model_rkv = AutoModelForCausalLM.from_pretrained(MODEL_ID, **_model_kwargs)
if not HAS_CUDA:
    model_rkv = model_rkv.to(DEVICE)
model_rkv.eval()
model_rkv.config.use_cache = True

device = next(model_rkv.parameters()).device

N_LAYERS   = model_rkv.config.num_hidden_layers
N_Q_HEADS  = model_rkv.config.num_attention_heads
N_KV_HEADS = int(getattr(model_rkv.config, "num_key_value_heads", N_Q_HEADS))
HEAD_DIM   = model_rkv.config.hidden_size // N_Q_HEADS
KV_GROUP   = N_Q_HEADS // N_KV_HEADS
print(f"model_rkv ready | layers={N_LAYERS} q_heads={N_Q_HEADS} kv_heads={N_KV_HEADS} "
      f"head_dim={HEAD_DIM} group={KV_GROUP} | attn: eager (required by RocketKV)")


## Helper Functions

In [ ]:
# Block - sync_if_cuda/clear_memory: used across every timed inference
# loop in this notebook (GSM8K, ARC-Challenge, HellaSwag) for timing-safe
# GPU synchronization and between-dataset memory cleanup.


def sync_if_cuda():
    if HAS_CUDA:
        torch.cuda.synchronize()


def clear_memory():
    gc.collect()
    if HAS_CUDA:
        torch.cuda.empty_cache()

In [ ]:
def seeded_subset(items, max_samples, seed=SHARED_SEED):
    """Select a reproducible random subset, then restore source order.

    A fresh RNG is created on every call, so running notebook sections in a
    different order cannot change which examples are selected.
    """
    items = list(items)

    sample_count = min(int(max_samples), len(items))

    selected_indices = sorted(
        random.Random(int(seed)).sample(range(len(items)), sample_count)
    )

    return [items[index] for index in selected_indices], selected_indices

In [ ]:
def robust_call(fn, *args, max_retries=5, backoff_sec=5, desc="operation", on_retry=None, **kwargs):
    """Retries fn(*args, **kwargs) on any exception, up to max_retries times,
    waiting backoff_sec between attempts -- guards dataset downloads against
    transient network failures (e.g. IncompleteRead/ChunkedEncodingError)
    rather than letting one flaky connection kill the whole notebook run.
    If on_retry is given, it's called (no args) after each failure, before
    the next attempt -- e.g. clear_hf_dataset_cache, so a retry that hit a
    stuck/corrupted partial download actually starts fresh instead of
    resuming (and re-breaking at) the same point every time."""
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_err = e
            _msg = f"  {desc}: attempt {attempt}/{max_retries} failed ({e!r})"
            if attempt < max_retries:
                _msg += f", retrying in {backoff_sec}s..."
            print(_msg)
            if attempt < max_retries:
                if on_retry is not None:
                    on_retry()
                time.sleep(backoff_sec)
    raise last_err

## RocketKV engine

Two stages implemented faithfully to arXiv:2502.14051 Section 3 / Algorithm 1, as an eager-attention path matching this harness. **Stage 1 (SnapKV eviction)** runs as an end-of-prefill cache prune (the same in-family mechanism H2O uses) and is the tested part. **Stage 2 (HSA)** is query-dependent per layer and therefore lives in a monkeypatched attention forward; it is a faithful transcription that has NOT been executed against the reference and needs a GPU shakeout -- gated by `HSA_ENABLED`. With `HSA_ENABLED=False` the notebook runs stage-1-only (a legitimate SnapKV-eviction path) end-to-end today.

In [ ]:
# Block - RocketKV sizing + auxiliary-storage utilities.
# All sizes derive from the paper. For a sequence length S and total token
# budget t, the total compression ratio is c = S / t. If c <= 1 (budget >=
# sequence length) NOTHING is compressed -- this is the no-op regime that makes
# the reduce-to-baseline invariant hold, and it is the common case on these
# short datasets (see the top-of-notebook caveat).

HSA_ENABLED = True   # False -> stage-1-only (pure SnapKV eviction), fully runnable today.
                     # True  -> also run stage-2 HSA (monkeypatched attention; VALIDATE ON GPU).

import torch.nn.functional as F


def rocketkv_split_factor(c):
    """Adaptive compression decomposition (paper 3.6): r = min(0.2 + 0.06*log2(c), 0.8),
    clamped to [R_MIN, R_MAX]. c is split into c^r (stage 1) and c^(1-r) (stage 2)."""
    if c <= 1.0:
        return ROCKETKV_R_MIN  # unused in the no-op case, but keep well-defined
    r = ROCKETKV_R_MIN + 0.06 * math.log2(c)
    return float(min(max(r, ROCKETKV_R_MIN), ROCKETKV_R_MAX))


def rocketkv_stage_sizes(S, t):
    """Return (stage1_kept, page_size, head_keep_k1, stage2_tokens_k2).
    stage1_kept : # prompt tokens SnapKV keeps (permanent eviction target).
    page_size   : sequence-dim page size for HSA Kmax/Kmin (stage 2).
    head_keep_k1: # head-dim positions kept for HSA top-k1 estimation.
    stage2_tokens_k2 : # tokens finally attended by HSA sparse attention.
    When c<=1, returns sizes that mean 'keep everything' (no-op)."""
    c = S / max(t, 1)
    if c <= 1.0:
        return S, 1, HEAD_DIM, S          # no compression at all
    r = rocketkv_split_factor(c)
    stage1_kept = max(1, math.ceil(S / (c ** r)))          # ~ S / c^r
    c2 = c ** (1.0 - r)                                     # stage-2 ratio
    per_dim = math.sqrt(c2)                                 # split evenly seq/head
    page_size = max(HSA_MIN_PAGE_SIZE, math.ceil(per_dim))
    head_ratio = c2 / page_size                             # remaining ratio -> head dim
    head_keep_k1 = max(1, math.ceil(HEAD_DIM / max(head_ratio, 1.0)))
    stage2_tokens_k2 = max(1, min(stage1_kept, t))          # final attended tokens ~ budget
    return stage1_kept, page_size, head_keep_k1, stage2_tokens_k2


def build_pages_maxmin(k_layer, page_size):
    """k_layer: [n_kv_heads, n_kept, head_dim] -> (kmax, kmin) each
    [n_kv_heads, n_pages, head_dim], element-wise max/min over each page along
    the sequence dim (Quest-style, but stored head-aligned as in HSA)."""
    n_kv, n, hd = k_layer.shape
    n_pages = math.ceil(n / page_size)
    pad = n_pages * page_size - n
    if pad:
        kpad = torch.cat([k_layer,
                          torch.full((n_kv, pad, hd), float("-inf"), dtype=k_layer.dtype, device=k_layer.device)], dim=1)
        kmax = kpad.view(n_kv, n_pages, page_size, hd).max(dim=2).values
        kpad_min = torch.cat([k_layer,
                          torch.full((n_kv, pad, hd), float("inf"), dtype=k_layer.dtype, device=k_layer.device)], dim=1)
        kmin = kpad_min.view(n_kv, n_pages, page_size, hd).min(dim=2).values
    else:
        kmax = k_layer.view(n_kv, n_pages, page_size, hd).max(dim=2).values
        kmin = k_layer.view(n_kv, n_pages, page_size, hd).min(dim=2).values
    return kmax, kmin


def _tensor_bytes(t):
    return t.numel() * t.element_size()


def resolve_token_budget(prompt_len, level_value, mode):
    """Map a 'level' to an absolute per-group token budget t for one prompt.
    proportional: level_value is the compression ratio c -> t = ceil(prompt_len / c).
    absolute:     level_value IS the token budget t (paper-matching / validation)."""
    if mode == "proportional":
        return max(1, math.ceil(prompt_len / float(level_value)))
    return int(level_value)


In [ ]:
# Block - RocketKV engine core.
# Integration strategy (mirrors H2O's in-family cache-manipulation, plus a
# monkeypatch that stage 2 genuinely requires):
#   * cache_to_legacy/get_cache_tokens : same helpers as the H2O notebook.
#   * STAGE 1 (SnapKV eviction) : after the prefill forward (output_attentions
#     =True) we score every prompt KV position from the observation window's
#     attention, pool it, keep the top `stage1_kept` (union the observation
#     window itself, which is always retained), and PRUNE the cache to those
#     survivors. This is exactly the eviction-by-cache-manipulation pattern H2O
#     uses and is the tested-in-structure part of this notebook.
#   * STAGE 2 (HSA) : query-dependent, per layer, per decode step -> it cannot
#     be done by pruning the cache between forwards (we don't have the layers'
#     queries out here). It is therefore a monkeypatch on LlamaAttention that,
#     at decode, approximates top-k tokens (paged Kmax/Kmin along seq + top-k1
#     along head) and sparse-attends. THIS PATCH IS A FAITHFUL TRANSCRIPTION OF
#     ALGORITHM 1 THAT HAS NOT BEEN RUN AGAINST THE REFERENCE -- validate on GPU.
#     If it fails to install on this transformers version, we warn and fall back
#     to stage-1-only (HSA_ENABLED effectively False).

def cache_to_legacy(pkv):
    if pkv is None: return None
    if isinstance(pkv, tuple): return pkv
    if hasattr(pkv, "to_legacy_cache"): return pkv.to_legacy_cache()
    raise TypeError(f"Unsupported cache type: {type(pkv)}")

def get_cache_tokens(pkv):
    pkv = cache_to_legacy(pkv)
    return 0 if pkv is None else int(pkv[0][0].shape[2])


# ---------------- STAGE 1: SnapKV eviction (per GQA group, pooled) ----------------
def snapkv_survivor_indices(attentions, prompt_len, stage1_kept):
    """attentions: tuple of [1, n_q_heads, q_len, kv_len] per layer (from the
    prefill forward with output_attentions=True). Returns a list (per layer) of
    LongTensor survivor indices into [0, prompt_len). The last SNAPKV_OBS_WINDOW
    positions (the observation window / most-recent tokens) are always kept."""
    obs = min(SNAPKV_OBS_WINDOW, prompt_len)
    if stage1_kept >= prompt_len or obs >= prompt_len:
        # No-op: either the budget already covers the whole prompt, or the
        # observation window itself already covers the whole prompt (a short
        # prompt -- e.g. a zero-shot ARC/HellaSwag context under
        # SNAPKV_OBS_WINDOW tokens). In the second case there is no "prefix"
        # region left to score/evict from (score[: prompt_len - obs] would be
        # a zero-length tensor, which F.max_pool1d cannot pool over), so
        # every position survives.
        keep_all = torch.arange(prompt_len, device=device)
        return [keep_all for _ in attentions]
    n_extra = max(0, stage1_kept - obs)
    survivors = []
    for att in attentions:
        # attention FROM the observation window TO all positions, summed over the
        # window's query rows, then reduced query-heads -> KV groups (Ada-KV style)
        a = att[0, :, prompt_len - obs:, :]                      # [n_q_heads, obs, kv_len]
        score = a.sum(dim=1)                                     # [n_q_heads, kv_len]
        score = score.view(N_KV_HEADS, KV_GROUP, -1).sum(dim=1)  # [n_kv_heads, kv_len]
        score = score.sum(dim=0)                                 # [kv_len] shared across group
        # never let the observation window compete; score only the prefix region
        prefix = score[: prompt_len - obs].clone()
        # pooling so neighbours of a selected token survive (SnapKV completeness)
        k = SNAPKV_KERNEL_SIZE
        pooled = F.max_pool1d(prefix.float().view(1, 1, -1), kernel_size=k,
                              stride=1, padding=k // 2).view(-1)[: prefix.numel()]
        topk = min(n_extra, pooled.numel())
        sel = torch.topk(pooled, topk).indices if topk > 0 else torch.empty(0, dtype=torch.long, device=device)
        obs_idx = torch.arange(prompt_len - obs, prompt_len, device=device)
        survivors.append(torch.sort(torch.cat([sel.to(device), obs_idx]))[0])
    return survivors


def prune_pkv_to_survivors(pkv, survivor_idx):
    """Gather each layer's K/V at that layer's survivor indices -> pruned legacy pkv."""
    pkv = cache_to_legacy(pkv)
    out = []
    for layer, (k, v) in enumerate(pkv):
        idx = survivor_idx[layer]
        out.append((k[:, :, idx, :].contiguous(), v[:, :, idx, :].contiguous()))
    return tuple(out)


def rocketkv_cache_bytes(pkv, page_size, hsa_on):
    """Resident KV bytes for RocketKV = survivor K+V (dense) + (if HSA on) the
    auxiliary paged Kmax/Kmin storage. Mirrors the paper's storage model
    (1/c^r + 2/c^((1-r)/2)); measured from real tensor sizes."""
    pkv = cache_to_legacy(pkv)
    total = 0
    for k, v in pkv:
        total += _tensor_bytes(k) + _tensor_bytes(v)
        if hsa_on:
            n = k.shape[2]
            n_pages = math.ceil(n / page_size)
            # kmax + kmin, each [n_kv_heads, n_pages, head_dim], same dtype as K
            total += 2 * (k.shape[1] * n_pages * k.shape[3]) * k.element_size()
    return int(total)


# ---------------- STAGE 2: HSA monkeypatch (UNVALIDATED -- see header) ----------------
# Kept minimal and guarded. Installs a wrapper on the eager attention that, at
# decode (q_len==1) and when RKV_STATE['hsa'] is on, restricts attention to an
# approximate top-k2 token set per layer. Selection uses this layer's query
# against paged Kmax/Kmin (seq dim) with top-k1 head-dim positions (SparQ-style).
RKV_STATE = {"hsa": False, "page_size": 1, "k1": HEAD_DIM, "k2": 10**9, "layer": 0}
_ORIG_ATTN_FORWARD = {"fn": None}

def install_hsa_patch():
    """Attempt to monkeypatch LlamaAttention.forward for HSA. Faithful to
    Algorithm 1 but UNTESTED against the reference; on any failure we warn and
    leave stage-1-only behaviour intact."""
    try:
        import transformers.models.llama.modeling_llama as ml
        if _ORIG_ATTN_FORWARD["fn"] is not None:
            return True
        _ORIG_ATTN_FORWARD["fn"] = ml.LlamaAttention.forward

        def hsa_forward(self, hidden_states, attention_mask=None, position_ids=None,
                        past_key_value=None, output_attentions=False, use_cache=False,
                        cache_position=None, position_embeddings=None, **kw):
            # Only intervene at single-token decode with HSA active; otherwise
            # defer to the original implementation (prefill, or HSA off).
            q_len = hidden_states.shape[1]
            if not (RKV_STATE["hsa"] and q_len == 1):
                return _ORIG_ATTN_FORWARD["fn"](
                    self, hidden_states, attention_mask=attention_mask,
                    position_ids=position_ids, past_key_value=past_key_value,
                    output_attentions=output_attentions, use_cache=use_cache,
                    cache_position=cache_position, position_embeddings=position_embeddings, **kw)
            # ---- run original to get the standard cache update + projections,
            # then OVERRIDE the attention output with an HSA-sparse computation.
            # (We recompute q/k/v here from the layer's own projections.) ----
            bsz, _, _ = hidden_states.size()
            q = self.q_proj(hidden_states)
            k = self.k_proj(hidden_states)
            v = self.v_proj(hidden_states)
            q = q.view(bsz, 1, self.num_heads, self.head_dim).transpose(1, 2)
            k = k.view(bsz, 1, self.num_key_value_heads, self.head_dim).transpose(1, 2)
            v = v.view(bsz, 1, self.num_key_value_heads, self.head_dim).transpose(1, 2)
            cos, sin = position_embeddings
            q, k = ml.apply_rotary_pos_emb(q, k, cos, sin)
            if past_key_value is not None:
                cache_kwargs = {"sin": sin, "cos": cos, "cache_position": cache_position}
                k, v = past_key_value.update(k, v, self.layer_idx, cache_kwargs)
            # k,v now [b, n_kv, n_all, hd]; q [b, n_q, 1, hd]
            kf, vf = k[0], v[0]                      # [n_kv, n_all, hd]
            qf = q[0]                                # [n_q, 1, hd]
            n_all = kf.shape[1]
            k2 = min(RKV_STATE["k2"], n_all)
            page_size = RKV_STATE["page_size"]
            k1 = min(RKV_STATE["k1"], self.head_dim)
            # per KV group query (sum |q| over group heads) for head-dim topk
            qg = qf.view(self.num_key_value_heads, self.num_heads // self.num_key_value_heads, 1, self.head_dim)
            qabs = qg.abs().sum(dim=1)              # [n_kv, 1, hd]
            head_idx = torch.topk(qabs.squeeze(1), k1, dim=-1).indices   # [n_kv, k1]
            qsum = qg.sum(dim=1).squeeze(1)         # [n_kv, hd] (signed, for max/min pick)
            # paged max/min over the current keys
            kmax, kmin = build_pages_maxmin(kf, page_size)   # [n_kv, n_pages, hd]
            sel_out = torch.empty(self.num_heads, 1, self.head_dim, dtype=qf.dtype, device=qf.device)
            for h in range(self.num_key_value_heads):
                hi = head_idx[h]                                   # [k1]
                qs = qsum[h, hi]                                   # [k1]
                pmax = kmax[h][:, hi]; pmin = kmin[h][:, hi]       # [n_pages, k1]
                approx = torch.where(qs >= 0, qs * pmax, qs * pmin).sum(dim=-1)  # [n_pages]
                n_pages = approx.shape[0]
                # expand page scores to token scores, take top-k2 tokens
                tok_scores = approx.repeat_interleave(page_size)[:n_all]
                sel = torch.topk(tok_scores, k2).indices          # [k2]
                ksel = kf[h][sel]; vsel = vf[h][sel]              # [k2, hd]
                for g in range(self.num_heads // self.num_key_value_heads):
                    qh = qf[h * (self.num_heads // self.num_key_value_heads) + g]  # [1, hd]
                    a = (qh @ ksel.t()) / math.sqrt(self.head_dim)               # [1, k2]
                    a = torch.softmax(a.float(), dim=-1).to(vsel.dtype)
                    sel_out[h * (self.num_heads // self.num_key_value_heads) + g] = a @ vsel
            attn_out = sel_out.transpose(0, 1).reshape(bsz, 1, self.num_heads * self.head_dim)
            attn_out = self.o_proj(attn_out)
            return (attn_out, None, past_key_value) if use_cache else (attn_out, None)

        ml.LlamaAttention.forward = hsa_forward
        print("HSA monkeypatch installed (UNVALIDATED -- verify on GPU before trusting stage-2 output).")
        return True
    except Exception as e:
        print(f"HSA patch failed to install ({e!r}); falling back to stage-1-only.")
        return False

def uninstall_hsa_patch():
    if _ORIG_ATTN_FORWARD["fn"] is not None:
        import transformers.models.llama.modeling_llama as ml
        ml.LlamaAttention.forward = _ORIG_ATTN_FORWARD["fn"]
        _ORIG_ATTN_FORWARD["fn"] = None

_HSA_OK = install_hsa_patch() if HSA_ENABLED else False


In [ ]:
# Block - RocketKV prefill (with stage-1 eviction) and decode step. Timing
# definitions are IDENTICAL to the H2O/baseline notebooks: TTFT = time to the
# first token's logits (the prefill forward); total = whole generation; TBT =
# (total - TTFT)/(n_generated-1). n_generated counts every forward-produced
# token incl. the final EOS step. Memory is peak KV-CACHE BYTES across BOTH
# prefill and decode, measured the same tensor-byte way as the H2O/KVQuant/
# baseline cache metrics. RocketKV's high-water mark is at END OF PREFILL: the
# full dense prompt cache exists before stage-1 eviction (RocketKV must see the
# whole prompt's attention to know what to evict), so that dense value is the
# peak and it is budget-INDEPENDENT across levels.

@torch.no_grad()
def rocketkv_prefill(input_ids, t):
    prompt_len = input_ids.shape[1]
    stage1_kept, page_size, k1, k2 = rocketkv_stage_sizes(prompt_len, t)
    c = prompt_len / max(t, 1)
    hsa = bool(_HSA_OK and c > 1.0)   # no-op regime -> full attention -> reduces to baseline
    RKV_STATE.update(hsa=False)       # prefill always runs full (patch defers on q_len>1 anyway)
    outputs = model_rkv(input_ids=input_ids, use_cache=True,
                        output_attentions=True, return_dict=True)
    last_logits = outputs.logits[:, -1, :]
    # Peak KV-cache is HERE: full dense prompt cache before stage-1 eviction.
    # Same tensor-byte accounting (numel * element_size over K,V, all layers).
    _dense = cache_to_legacy(outputs.past_key_values)
    prefill_peak_bytes = int(sum(k.numel() * k.element_size() + v.numel() * v.element_size()
                                 for k, v in _dense))
    # STAGE 1: evict prompt cache to survivors
    survivor_idx = snapkv_survivor_indices(outputs.attentions, prompt_len, stage1_kept)
    pkv = prune_pkv_to_survivors(outputs.past_key_values, survivor_idx)
    sizes = (stage1_kept, page_size, k1, k2)
    return last_logits, pkv, sizes, hsa, prefill_peak_bytes

@torch.no_grad()
def rocketkv_step(token_tensor, abs_pos, pkv, sizes, hsa):
    stage1_kept, page_size, k1, k2 = sizes
    RKV_STATE.update(hsa=hsa, page_size=page_size, k1=k1, k2=k2)
    cache_len = get_cache_tokens(pkv)
    attn_mask = torch.ones((1, cache_len + 1), dtype=torch.long, device=device)
    pos = torch.tensor([[abs_pos]], dtype=torch.long, device=device)
    out = model_rkv(input_ids=token_tensor, past_key_values=pkv,
                    attention_mask=attn_mask, position_ids=pos,
                    use_cache=True, return_dict=True)
    RKV_STATE.update(hsa=False)
    return out.logits[:, -1, :], cache_to_legacy(out.past_key_values)


## GSM8K

In [ ]:
# Block - GSM8K loading: combine all splits, then random sampling.
# Load train + test splits, build the complete list of valid question/answer
# pairs, then select up to 2,048 examples with seed 42. This creates a
# reproducible random sample across the entire GSM8K dataset.

def extract_gsm8k_gold_answer(answer_text):
    match = re.search(r"####\s*(-?[0-9][0-9,]*\.?[0-9]*)", answer_text)
    if not match:
        return None
    try:
        return float(match.group(1).replace(",", ""))
    except ValueError:
        return None

gsm8k_train = robust_call(
    load_dataset,
    "gsm8k",
    "main",
    split="train",
    desc="GSM8K train load",
    on_retry=lambda: clear_hf_dataset_cache("gsm8k"),
)

gsm8k_test = robust_call(
    load_dataset,
    "gsm8k",
    "main",
    split="test",
    desc="GSM8K test load",
    on_retry=lambda: clear_hf_dataset_cache("gsm8k"),
)

# Combine all official GSM8K splits
gsm8k_all = list(gsm8k_train) + list(gsm8k_test)
all_gsm8k_pairs = []

for item in gsm8k_all:
    gold = extract_gsm8k_gold_answer(item["answer"])
    if gold is not None:
        all_gsm8k_pairs.append({
            "question": item["question"],
            "gold": gold,
            "gold_text": item["answer"],
        })

gsm8k_qa_pairs, gsm8k_selected_indices = seeded_subset(
    all_gsm8k_pairs,
    QA_EVAL_SAMPLES,
    SHARED_SEED,
)

print(
    f"GSM8K: {len(all_gsm8k_pairs)} valid questions available; "
    f"selected {len(gsm8k_qa_pairs)} random questions "
    f"(requested {QA_EVAL_SAMPLES}, seed={SHARED_SEED})"
)

print(
    "GSM8K selected valid-item indices (first 20):",
    gsm8k_selected_indices[:20]
)

In [ ]:
# Block - GSM8K generation with RocketKV. Same hand-rolled prefill+greedy-decode
# structure and metric/CSV schema as the H2O/baseline notebooks; only the engine
# differs (stage-1 eviction at prefill + stage-2 HSA per decode step).

def _extract_final_number(text):
    m = re.search(r"####\s*(-?[0-9][0-9,]*\.?[0-9]*)", text)
    if m:
        num_str = m.group(1)
    else:
        nums = re.findall(r"-?[0-9][0-9,]*\.?[0-9]*", text)
        if not nums: return None
        num_str = nums[-1]
    num_str = num_str.replace(",", "").rstrip(".")
    try: return float(num_str)
    except ValueError: return None


@torch.no_grad()
def generate_gsm8k_rocketkv(question, level_value, mode=None):
    prompt = GSM8K_FEWSHOT_PREFIX + f"\nQuestion: {question.strip()}\nAnswer:"
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_len = enc["input_ids"].shape[1]
    t = resolve_token_budget(prompt_len, level_value, mode or BUDGET_MODE)

    sync_if_cuda(); gen_start = time.perf_counter()
    last_logits, pkv, sizes, hsa, peak_bytes = rocketkv_prefill(enc["input_ids"], t)
    next_id = int(last_logits.argmax(dim=-1)[0].item())
    sync_if_cuda(); ttft_sec = time.perf_counter() - gen_start

    page_size = sizes[1]
    generated_ids = []; n_forward_tokens = 1
    hash_streak = 0; in_answer_span = False; answer_done = False
    answer_token_ids = []; answer_logits = []

    for step in range(GSM8K_MAX_NEW_TOKENS):
        if next_id == tokenizer.eos_token_id: break
        generated_ids.append(next_id)
        if not answer_done:
            tok_text = tokenizer.decode([next_id])
            if in_answer_span:
                if "#" in tok_text: answer_done = True
                else:
                    answer_token_ids.append(next_id); answer_logits.append(last_logits.detach())
            else:
                for ch in tok_text:
                    hash_streak = hash_streak + 1 if ch == "#" else 0
                if hash_streak >= 4: in_answer_span = True
        if "\n" in tokenizer.decode([next_id]):        # early stop: halt after the #### answer line
            _run_txt = tokenizer.decode(generated_ids)
            if "####" in _run_txt and "\n" in _run_txt.split("####", 1)[-1]:
                break
        if step == GSM8K_MAX_NEW_TOKENS - 1: break
        token_tensor = torch.tensor([[next_id]], dtype=torch.long, device=device)
        last_logits, pkv = rocketkv_step(token_tensor, prompt_len + step, pkv, sizes, hsa)
        next_id = int(last_logits.argmax(dim=-1)[0].item())
        n_forward_tokens += 1
        peak_bytes = max(peak_bytes, rocketkv_cache_bytes(pkv, page_size, hsa))

    sync_if_cuda(); gen_end = time.perf_counter()
    total_latency_sec = gen_end - gen_start

    if answer_token_ids:
        nll_sum = 0.0
        for tok_id, logits in zip(answer_token_ids, answer_logits):
            lp = torch.log_softmax(logits[0].float(), dim=-1); nll_sum += -lp[tok_id].item()
        perplexity = math.exp(min(nll_sum / len(answer_token_ids), 50.0))
    else:
        perplexity = None

    gen_text = tokenizer.decode(generated_ids, skip_special_tokens=True).split("Question:")[0]
    n_generated = n_forward_tokens
    if len(generated_ids) == 0: ttft_sec = total_latency_sec
    tbt_sec = (total_latency_sec - ttft_sec) / max(n_generated - 1, 1)
    return {
        "full_prompt": prompt, "prefill_tokens": prompt_len, "generated_tokens": len(generated_ids), "gen_text": gen_text, "ttft_sec": ttft_sec, "tbt_sec": tbt_sec,
        "total_latency_sec": total_latency_sec, "total_tokens": prompt_len + n_generated,
        "peak_memory_bytes": peak_bytes, "perplexity": perplexity,
    }


def evaluate_gsm8k_rocketkv(qa_pairs, method_label, level_value):
    clear_memory()
    correct = total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values = [], [], [], [], []
    per_question_records = []
    N_PREVIEW_QUESTIONS = 1000
    for q_idx, qa in enumerate(tqdm(qa_pairs, desc=f"GSM8K | {method_label}")):
        result = generate_gsm8k_rocketkv(qa["question"], level_value)
        pred = _extract_final_number(result["gen_text"])
        is_correct = pred is not None and abs(pred - qa["gold"]) < 1e-4
        correct += int(is_correct); total += 1
        if q_idx < N_PREVIEW_QUESTIONS:
            print(f"\n--- GSM8K | {method_label} | question {q_idx} preview ---")
            print(f"Question:    {qa['question']}")
            print(f"Generated:   {result['gen_text'].strip()}")
            print(f"Gold answer: {qa['gold']} | Predicted: {pred} | Correct: {is_correct}")
        ttft_values.append(result["ttft_sec"]); tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"]); peak_mem_values.append(result["peak_memory_bytes"])
        if result["perplexity"] is not None: ppl_values.append(result["perplexity"])
        per_question_records.append({
            "question_index": q_idx,
            "full_prompt": result["full_prompt"],
            "prefill_tokens": result["prefill_tokens"],
            "generated_output": result["gen_text"],
            "generated_tokens": result["generated_tokens"],
            "gold_answer": qa["gold"],
            "predicted_answer": pred,
            "correct": int(is_correct),
            "perplexity": result["perplexity"],
            "ttft_sec": result["ttft_sec"],
            "tbt_sec": result["tbt_sec"],
            "total_latency_sec": result["total_latency_sec"],
            "peak_memory_mb": result["peak_memory_bytes"] / 1024**2,
        })
    accuracy = correct / max(total, 1)
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")
    os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)
    _p = f"/content/drive/MyDrive/KVQuant_v3_Results/{method_label}_gsm8k_per_prompt.csv"
    pd.DataFrame(per_question_records).to_csv(_p, index=False)
    print(f"Saved {len(per_question_records)} per-question GSM8K rows to {_p}")
    return {
        "dataset": "GSM8K", "method": method_label, "perplexity": avg_ppl, "accuracy": accuracy,
        "ttft_sec": sum(ttft_values)/len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values)/len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values)/len(latency_values) if latency_values else float("nan"),
        "peak_memory_mb": max(peak_mem_values)/1024**2 if peak_mem_values else 0.0,
        "average_memory_mb": (sum(peak_mem_values)/len(peak_mem_values)/1024**2) if peak_mem_values else 0.0,
    }


In [ ]:
# ============================================================================
# Shared multiple-choice (MC) scoring machinery -- ARC-Challenge and
# HellaSwag are scored via teacher-forced per-choice likelihood, matching the
# large_sample_implementations family's methodology exactly (not
# generate-and-extract). Each choice's CONTEXT is prefilled through
# rocketkv_prefill (stage-1 SnapKV eviction sees the full context's
# attention, exactly like a GSM8K prompt's prefill), then the choice's own
# continuation tokens are teacher-forced one at a time through rocketkv_step
# (stage-2 HSA applies exactly as it does during real decoding), and every
# continuation token's cross-entropy is summed into that choice's NLL. The
# model is never asked to generate anything for these two datasets -- there
# is no few-shot prefix either; both datasets are zero-shot. GSM8K is
# untouched -- it keeps its own separate generative grading
# (generate_gsm8k_rocketkv, above).
#
# lm_eval_encode_pair -- identical to the large_sample_implementations family
# (same joint-tokenization + splitting logic, matching lm-evaluation-harness).
#
# score_mc_choice_rocketkv(prompt, choice, level_value, mode=None)
#     The choice's own CONTEXT length (not context+continuation) is the
#     "prompt length" RocketKV budgets from -- exactly like every other
#     dataset in this notebook derives its token budget t from its own
#     prompt's length. Memory follows the family convention: RocketKV's
#     high-water mark is at end of (context) prefill, before stage-1
#     eviction, so that dense pre-eviction figure is reported as this
#     choice's peak.
#
# score_mc_question_rocketkv(prompt, choices, gold_index, level_value, mode=None)
#     Same aggregation as the baseline/H2O/KVQuant notebooks: raw/normalized
#     accuracy, perplexity from the gold choice only, TTFT = mean across
#     choices, TBT = weighted mean across every choice's decode steps, total
#     latency = SUM across choices, peak memory = MAX across choices.
# ============================================================================


def lm_eval_encode_pair(context, choice):
    context = str(context)
    continuation = " " + str(choice)

    n_spaces = len(context) - len(context.rstrip())
    if n_spaces > 0:
        continuation = context[-n_spaces:] + continuation
        context = context[:-n_spaces]

    if not context:
        raise ValueError("MC context cannot be empty.")

    whole_ids = tokenizer(context + continuation, add_special_tokens=True)["input_ids"]
    context_ids = tokenizer(context, add_special_tokens=True)["input_ids"]
    continuation_ids = whole_ids[len(context_ids):]

    if not context_ids:
        raise ValueError("Context tokenization produced no tokens.")
    if not continuation_ids:
        raise ValueError(f"Continuation tokenization produced no tokens. Context={context!r}, choice={choice!r}")

    return context_ids, continuation_ids

@torch.no_grad()
def score_mc_choice_rocketkv(prompt, choice, level_value, mode=None):
    context_ids, continuation_ids = lm_eval_encode_pair(prompt, choice)
    n_context = len(context_ids)
    t = resolve_token_budget(n_context, level_value, mode or BUDGET_MODE)

    loss_fct = nn.CrossEntropyLoss()
    nll_sum = 0.0
    scored = 0
    step_times = []

    context_tensor = torch.tensor([context_ids], dtype=torch.long, device=device)

    sync_if_cuda()
    t0 = time.perf_counter()
    last_logits, pkv, sizes, hsa, peak_bytes = rocketkv_prefill(context_tensor, t)
    sync_if_cuda()
    step_times.append(time.perf_counter() - t0)

    # Score continuation[0] using the prefill's own last_logits -- no extra
    # forward needed, exactly like the GSM8K/MCQ generation functions reuse
    # the prefill's next_id.
    target0 = torch.tensor([continuation_ids[0]], device=device)
    nll_sum += loss_fct(last_logits, target0).float().item()
    scored += 1

    for i in range(len(continuation_ids) - 1):
        token_tensor = torch.tensor([[continuation_ids[i]]], dtype=torch.long, device=device)
        abs_pos = n_context + i

        sync_if_cuda()
        ts = time.perf_counter()
        step_logits, pkv = rocketkv_step(token_tensor, abs_pos, pkv, sizes, hsa)
        sync_if_cuda()
        step_times.append(time.perf_counter() - ts)

        target = torch.tensor([continuation_ids[i + 1]], device=device)
        nll_sum += loss_fct(step_logits, target).float().item()
        scored += 1

    ttft_sec = step_times[0]
    decode_time_sum = sum(step_times[1:])
    decode_steps = len(step_times) - 1
    total_latency_sec = sum(step_times)

    return {
        "nll_sum": nll_sum, "scored": scored,
        "ttft_sec": ttft_sec, "decode_time_sum": decode_time_sum, "decode_steps": decode_steps,
        "total_latency_sec": total_latency_sec, "peak_memory_bytes": peak_bytes,
        "choice_char_len": max(len(str(choice)), 1),
    }


@torch.no_grad()
def score_mc_question_rocketkv(prompt, choices, gold_index, level_value, mode=None):
    choice_results = [score_mc_choice_rocketkv(prompt, choice, level_value, mode) for choice in choices]

    normalized_nlls = [r["nll_sum"] / r["choice_char_len"] for r in choice_results]

    raw_prediction = int(min(range(len(choice_results)), key=lambda i: choice_results[i]["nll_sum"]))
    normalized_prediction = int(min(range(len(choice_results)), key=lambda i: normalized_nlls[i]))

    gold_result = choice_results[gold_index]
    gold_mean_nll = gold_result["nll_sum"] / max(gold_result["scored"], 1)

    total_decode_time = sum(r["decode_time_sum"] for r in choice_results)
    total_decode_steps = sum(r["decode_steps"] for r in choice_results)

    return {
        "raw_prediction": raw_prediction,
        "normalized_prediction": normalized_prediction,
        "raw_correct": int(raw_prediction == gold_index),
        "normalized_correct": int(normalized_prediction == gold_index),
        "perplexity": math.exp(min(gold_mean_nll, 50.0)),
        "ttft_sec": sum(r["ttft_sec"] for r in choice_results) / len(choice_results),
        "tbt_sec": (total_decode_time / total_decode_steps) if total_decode_steps > 0 else 0.0,
        "total_latency_sec": sum(r["total_latency_sec"] for r in choice_results),
        "peak_memory_bytes": max(r["peak_memory_bytes"] for r in choice_results),
    }


## Validation (reduce-to-baseline invariant)

In [ ]:
# Block - VALIDATION. The one invariant that IS checkable on short data:
# at a token budget >= the sequence length, RocketKV compresses NOTHING
# (stage 1 keeps all tokens, stage 2 is disabled in the no-op regime), so its
# greedy generation must reproduce a plain model.generate() call exactly. This
# validates the harness plumbing (eviction prune, cache feeding, position ids)
# end to end. It does NOT validate the HSA math -- that only activates at
# budgets < sequence length and has no short-data reference to check against
# (see the top-of-notebook caveat); flag it for a GPU shakeout separately.

_val_q = gsm8k_qa_pairs[0]["question"]
_val_prompt = GSM8K_FEWSHOT_PREFIX + f"\nQuestion: {_val_q.strip()}\nAnswer:"
_val_enc = tokenizer(_val_prompt, return_tensors="pt").to(device)

with torch.no_grad():
    _ref = model_rkv.generate(**_val_enc, max_new_tokens=GSM8K_MAX_NEW_TOKENS,
                              do_sample=False, pad_token_id=tokenizer.pad_token_id)
_ref_text = tokenizer.decode(_ref[0][_val_enc["input_ids"].shape[1]:], skip_special_tokens=True).split("Question:")[0]
_ref_pred = _extract_final_number(_ref_text)

_HUGE_BUDGET = 10**9   # >> any prompt length -> no-op regime
_hand = generate_gsm8k_rocketkv(_val_q, _HUGE_BUDGET, mode="absolute")
_hand_pred = _extract_final_number(_hand["gen_text"])

print("Gold:", gsm8k_qa_pairs[0]["gold"])
print("\n-- reference plain generate() --\n", _ref_text.strip(), "\npredicted:", _ref_pred)
print("\n-- RocketKV @ budget>=seqlen (should be identical) --\n", _hand["gen_text"].strip(), "\npredicted:", _hand_pred)
assert _ref_pred == _hand_pred, (
    f"Reduce-to-baseline FAILED (ref={_ref_pred}, rocketkv={_hand_pred}). At a budget >= sequence "
    "length RocketKV must equal full-KV. A mismatch is a plumbing bug (eviction/prune/position_ids), "
    "not fp noise -- fix before trusting any RocketKV number.")
print("\nVALIDATION PASSED: RocketKV reduces to the full-KV baseline at budget >= sequence length.")

# Quick engagement check (informational, no assert): at a small budget the
# method should actually compress -- show the stage sizes it picks.
_val_plen = _val_enc["input_ids"].shape[1]
print(f"(prompt length here = {_val_plen} tokens; BUDGET_MODE = {BUDGET_MODE})")
for _lvl in ROCKETKV_LEVELS:
    _t = resolve_token_budget(_val_plen, _lvl, BUDGET_MODE)
    s1, ps, k1, k2 = rocketkv_stage_sizes(_val_plen, _t)
    _c = _val_plen / _t
    _tag = f"ratio {_lvl}x" if BUDGET_MODE == "proportional" else f"budget {_lvl}"
    print(f"{_tag:>12s}: t={_t:5d}  c={_c:5.1f}  stage1_kept={s1:4d}  page_size={ps}  head_k1={k1}  stage2_k2={k2}"
          + ("   (NO-OP: budget>=prompt)" if _c <= 1 else ""))


## ARC-Challenge

In [ ]:
# Block - ARC-Challenge loading: ALL official splits (train + validation +
# test) combined, then random sampling -- mirrors GSM8K's own loading exactly
# (GSM8K combines train+test since it has no validation split; ARC-Challenge
# has all three, so all three go in). Every row across the combined pool gets
# the same validity filtering, then seeded_subset draws up to QA_EVAL_SAMPLES
# reproducibly. Scored by teacher-forced per-choice likelihood via
# score_mc_question_* (Helper Functions, above), zero-shot, not generation.


def load_arc_challenge_items():
    arc_train = robust_call(
        load_dataset, "allenai/ai2_arc", "ARC-Challenge", split="train",
        desc="ARC-Challenge train load", on_retry=lambda: clear_hf_dataset_cache("ai2_arc"),
    )
    arc_validation = robust_call(
        load_dataset, "allenai/ai2_arc", "ARC-Challenge", split="validation",
        desc="ARC-Challenge validation load", on_retry=lambda: clear_hf_dataset_cache("ai2_arc"),
    )
    arc_test = robust_call(
        load_dataset, "allenai/ai2_arc", "ARC-Challenge", split="test",
        desc="ARC-Challenge test load", on_retry=lambda: clear_hf_dataset_cache("ai2_arc"),
    )
    arc_all = list(arc_train) + list(arc_validation) + list(arc_test)

    valid_items = []
    for row in arc_all:
        labels = row["choices"]["label"]
        texts = row["choices"]["text"]
        answer_key = row["answerKey"]
        if answer_key not in labels:
            continue
        valid_items.append({
            "question": row["question"],
            "choices": list(zip(labels, texts)),
            "gold_label": answer_key,
        })

    selected_items, selected_indices = seeded_subset(
        valid_items,
        QA_EVAL_SAMPLES,
        SHARED_SEED,
    )
    print(
        f"ARC-Challenge: {len(valid_items)} valid questions available "
        f"(train+validation+test combined); selected {len(selected_items)} "
        f"random questions (requested {QA_EVAL_SAMPLES}, seed={SHARED_SEED})"
    )
    print("ARC-Challenge selected valid-item indices (first 20):", selected_indices[:20])
    return selected_items


arc_items = load_arc_challenge_items()


## HellaSwag

In [ ]:
# Block - HellaSwag loading: ALL LABELED official splits (train +
# validation) combined, then random sampling -- mirrors GSM8K's own loading
# as closely as HellaSwag allows: HellaSwag's test split ships unlabeled
# (label == -1, no gold answer to score against), so it cannot be included
# the way GSM8K's test split is; train + validation is the full labeled pool
# available. Every row across the combined pool gets the same validity
# filtering, then seeded_subset draws up to QA_EVAL_SAMPLES reproducibly.
# Scored by teacher-forced per-choice likelihood via score_mc_question_*
# (Helper Functions, above), zero-shot, not generation.


def hellaswag_preprocess(text):
    text = str(text).strip()
    text = text.replace(" [title]", ". ")
    text = re.sub(r"\[.*?\]", "", text)
    text = text.replace("  ", " ")
    return text


def _hs_valid(item):
    label = str(item.get("label", "")).strip()
    endings = item.get("endings", [])
    return label.isdigit() and len(endings) >= 2 and 0 <= int(label) < len(endings)


def load_hellaswag_items():
    hs_train = robust_call(
        load_dataset, "Rowan/hellaswag", split="train",
        desc="HellaSwag train load", on_retry=lambda: clear_hf_dataset_cache("hellaswag"),
    )
    hs_validation = robust_call(
        load_dataset, "Rowan/hellaswag", split="validation",
        desc="HellaSwag validation load", on_retry=lambda: clear_hf_dataset_cache("hellaswag"),
    )
    hs_all = list(hs_train) + list(hs_validation)

    processed_items = []
    for row in hs_all:
        if not _hs_valid(row):
            continue
        context = str(row["ctx_a"]) + " " + str(row["ctx_b"]).capitalize()
        prompt = hellaswag_preprocess(str(row["activity_label"]) + ": " + context)
        choices = [hellaswag_preprocess(choice) for choice in row["endings"]]
        processed_items.append({
            "prompt": prompt,
            "choices": choices,
            "gold_index": int(row["label"]),
        })

    selected_items, selected_indices = seeded_subset(
        processed_items,
        QA_EVAL_SAMPLES,
        SHARED_SEED,
    )
    print(
        f"HellaSwag: {len(processed_items)} valid examples available "
        f"(train+validation combined; test excluded -- unlabeled); "
        f"selected {len(selected_items)} random examples "
        f"(requested {QA_EVAL_SAMPLES}, seed={SHARED_SEED})"
    )
    print("HellaSwag selected example indices (first 20):", selected_indices[:20])
    return selected_items


hellaswag_items = load_hellaswag_items()


## RULER


In [ ]:
# ============================================================================
# RULER settings + single-needle-in-a-haystack (NIAH) example generation.
#
# RULER (Hsieh et al., "What's the Real Context Size of Your Long-Context
# Language Models?", arXiv:2404.06654) is a SYNTHETIC long-context benchmark
# GENERATOR, not a fixed dataset -- NVIDIA's own reference implementation
# builds examples per-tokenizer/per-length/per-seed rather than shipping one
# canonical pre-built dataset (there is no single official HF dataset
# covering 4K/8K/16K/32K for an arbitrary model). This notebook generates
# RULER's flagship task -- single-needle retrieval (niah_single) -- directly,
# using THIS model's own tokenizer for exact length control, following
# NVIDIA/RULER's own documented recipe (scripts/data/synthetic/niah.py /
# constants.py): a "noise" haystack (RULER's own filler-sentence haystack
# type, not an invented approximation) with one key/value "needle" sentence
# inserted at a random depth, and RULER's own official niah prompt template
# + answer_prefix (verbatim, type_needle_v="numbers").
#
# Length buckets: 4096 / 8192 / 16384 tokens (~683/683/682 samples, 2048
# total). 32768 is DELIBERATELY EXCLUDED: eager attention (required
# throughout this notebook family) materializes the full [heads, seq, seq]
# attention-score matrix, which at 32K tokens is on the order of 60-140GB
# for a SINGLE layer -- a risk shared identically by every compression
# method here (compression shrinks the KV CACHE, not the attention-score
# computation itself, so no method here is protected from it). 16384 is
# still non-trivial (~17GB/layer at bf16) -- if it OOMs on your GPU, shrink
# RULER_LENGTH_BUCKETS below; that is a hardware ceiling, not a code bug.
#
# Grading matches RULER's own convention: does the gold value string appear
# in the model's generated text (substring/recall match), not exact-string
# equality of the whole output.

import uuid

RULER_LENGTH_BUCKETS = [4096, 8192, 16384]
RULER_MAX_NEW_TOKENS = 128  # matches RULER's own niah task config (tokens_to_generate=128)

_ruler_base, _ruler_rem = divmod(QA_EVAL_SAMPLES, len(RULER_LENGTH_BUCKETS))
RULER_SAMPLES_PER_BUCKET = {
    length: _ruler_base + (1 if i < _ruler_rem else 0)
    for i, length in enumerate(RULER_LENGTH_BUCKETS)
}

# RULER's own "noise" haystack type (NVIDIA/RULER niah.py, haystack_type=
# "noise"): a fixed pool of filler sentences, repeated to reach the target
# length.
RULER_HAYSTACK_SENTENCES = [
    "The grass is green.",
    "The sky is blue.",
    "The sun is yellow.",
    "Here we go.",
    "There and back again.",
]
RULER_SENTENCE_TOKENS = {
    s: len(tokenizer(s, add_special_tokens=False)["input_ids"])
    for s in RULER_HAYSTACK_SENTENCES
}

# RULER's own official niah prompt template + answer_prefix (verbatim, from
# NVIDIA/RULER scripts/data/synthetic/constants.py's 'niah' task, with
# type_needle_v="numbers"), plus one explicit generation instruction so the
# model is told exactly how to format its answer -- matching this notebook
# family's convention of stating the expected output format explicitly
# (the same way GSM8K's few-shot prefix states "end with exactly this
# format: #### <final number>").
_RULER_HEADER = (
    "Some special magic numbers are hidden within the following text. Make "
    "sure to memorize them. I will quiz you about the numbers afterwards. "
    "Respond with only the magic number and nothing else.\n"
)
_RULER_QUERY_TAIL = (
    "\nWhat is the special magic number for {key} mentioned in the provided text?"
    "\nAnswer: The special magic number for {key} mentioned in the provided text is"
)


def _ruler_make_needle(rng):
    """RULER's own 'uuids' key type / 'numbers' value type (both official
    RULER type_needle options) -- no external word-list dependency needed."""
    key = str(uuid.UUID(int=rng.getrandbits(128)))
    value = str(rng.randint(1000000, 9999999))  # 7-digit number, RULER's default
    return key, value


def _ruler_build_item(rng, target_tokens):
    key, value = _ruler_make_needle(rng)
    needle_sentence = f"One of the special magic numbers for {key} is: {value}."

    # Build haystack sentences until the filler alone covers target_tokens
    # (a small, roughly-fixed header/query/answer-prefix/needle overhead
    # sits on top -- RULER's own lengths are nominal/approximate too, not
    # exact byte-for-byte token counts).
    sentences = []
    token_count = 0
    while token_count < target_tokens:
        sentence = rng.choice(RULER_HAYSTACK_SENTENCES)
        sentences.append(sentence)
        token_count += RULER_SENTENCE_TOKENS[sentence]

    # Insert the needle at a random depth (0-100% of the haystack), matching
    # RULER's own DEPTHS sampling (NVIDIA/RULER niah.py).
    insert_at = rng.randint(0, len(sentences))
    sentences.insert(insert_at, needle_sentence)
    context = " ".join(sentences)

    prompt = _RULER_HEADER + context + _RULER_QUERY_TAIL.format(key=key)

    return {"prompt": prompt, "key": key, "gold_value": value, "target_tokens": target_tokens}


def build_ruler_items():
    rng = random.Random(SHARED_SEED)
    items = []
    for target_tokens in RULER_LENGTH_BUCKETS:
        n = RULER_SAMPLES_PER_BUCKET[target_tokens]
        for _ in range(n):
            items.append(_ruler_build_item(rng, target_tokens))
    return items


ruler_items = build_ruler_items()
print(
    f"RULER: generated {len(ruler_items)} single-needle items -- "
    + ", ".join(f"{RULER_SAMPLES_PER_BUCKET[l]} @ {l} tokens" for l in RULER_LENGTH_BUCKETS)
)
_ruler_preview = ruler_items[0]
print(f"\nExample prompt (first {RULER_LENGTH_BUCKETS[0]}-token item), truncated:")
print(_ruler_preview["prompt"][:400] + " ...[haystack continues]... " + _ruler_preview["prompt"][-300:])
print(f"\nGold value: {_ruler_preview['gold_value']}")


## SQuAD1.1


In [ ]:
# ============================================================================
# SQuAD1.1 (rajpurkar/squad -- the standard v1.1 release, every question has
# an answer in its passage, unlike squad_v2's unanswerable questions):
# extractive reading comprehension. Loading combines train + validation
# (SQuAD ships no public-label test split), matching the GSM8K/ARC/HellaSwag
# convention of pooling every official labeled split before a seeded sample.
# Zero-shot, generation-based, graded with the OFFICIAL SQuAD metrics --
# Exact Match and F1 -- against the best of however many gold reference
# answers a question has (validation questions often carry more than one
# accepted answer; train questions usually carry exactly one).
# ============================================================================

import re
import string
from collections import Counter

SQUAD_MAX_NEW_TOKENS = 48  # SQuAD gold answers are short spans (usually 1-5 words)

SQUAD_HEADER = (
    "Answer the question based on the passage below. Respond with a short "
    "phrase copied directly from the passage that answers the question, and "
    "nothing else.\n\n"
)


def format_squad_prompt(item):
    return (
        SQUAD_HEADER
        + f"Passage: {item['context'].strip()}\n"
        + f"Question: {item['question'].strip()}\n"
        + "Answer:"
    )


def load_squad_items():
    squad_train = robust_call(
        load_dataset, "rajpurkar/squad", split="train",
        desc="SQuAD1.1 train load", on_retry=lambda: clear_hf_dataset_cache("squad"),
    )
    squad_validation = robust_call(
        load_dataset, "rajpurkar/squad", split="validation",
        desc="SQuAD1.1 validation load", on_retry=lambda: clear_hf_dataset_cache("squad"),
    )
    squad_all = list(squad_train) + list(squad_validation)

    valid_items = []
    for row in squad_all:
        texts = row.get("answers", {}).get("text", [])
        gold_answers = [str(t).strip() for t in texts if str(t).strip()]
        if not gold_answers or not str(row.get("context", "")).strip() or not str(row.get("question", "")).strip():
            continue
        valid_items.append({
            "context": row["context"],
            "question": row["question"],
            "gold_answers": gold_answers,
        })

    selected_items, selected_indices = seeded_subset(valid_items, QA_EVAL_SAMPLES, SHARED_SEED)
    print(
        f"SQuAD1.1: {len(valid_items)} valid questions available "
        f"(train+validation combined); selected {len(selected_items)} "
        f"random questions (requested {QA_EVAL_SAMPLES}, seed={SHARED_SEED})"
    )
    print("SQuAD1.1 selected valid-item indices (first 20):", selected_indices[:20])
    return selected_items


squad_items = load_squad_items()


# ---------------------------------------------------------------------------
# Official SQuAD1.1 scoring (Rajpurkar et al. 2016's own normalize_answer /
# exact_match_score / f1_score, reproduced verbatim): lowercase, drop
# punctuation, drop articles (a/an/the), collapse whitespace, then compare.
# Both metrics take the MAX over every gold reference answer available for
# that question.
# ---------------------------------------------------------------------------

def _squad_normalize(text):
    text = text.lower()
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = "".join(ch for ch in text if ch not in string.punctuation)
    text = " ".join(text.split())
    return text


def squad_exact_match(prediction, gold_answers):
    norm_pred = _squad_normalize(prediction)
    return max(int(norm_pred == _squad_normalize(g)) for g in gold_answers)


def squad_f1(prediction, gold_answers):
    pred_tokens = _squad_normalize(prediction).split()
    best = 0.0
    for g in gold_answers:
        gold_tokens = _squad_normalize(g).split()
        if len(pred_tokens) == 0 or len(gold_tokens) == 0:
            best = max(best, float(pred_tokens == gold_tokens))
            continue
        common = Counter(pred_tokens) & Counter(gold_tokens)
        num_same = sum(common.values())
        if num_same == 0:
            continue
        precision = num_same / len(pred_tokens)
        recall = num_same / len(gold_tokens)
        best = max(best, 2 * precision * recall / (precision + recall))
    return best


## Run all datasets across token budgets

In [ ]:
# Block - Shared method label + level, computed ONCE and independent of
# which dataset cell(s) you run below. Previously _lbl/_lvl/all_rows lived
# inside the GSM8K driver cell, which forced GSM8K to run first before
# ARC-Challenge or HellaSwag could even execute (they referenced _lbl/_lvl
# from that cell, and appended into its all_rows list). Every dataset driver
# cell below is now independently runnable -- run only the one(s) you want,
# in any order -- matching the "run just a subset of datasets" convention
# the Baseline/H2O/KVQuant notebooks already use.
_lvl = ROCKETKV_LEVELS[0]
_lbl = (f"rocketkv_ratio_{_lvl}x" if BUDGET_MODE == "proportional"
        else f"rocketkv_budget_{_lvl}") + ("" if _HSA_OK else "_stage1only")
print(f"Method label for this notebook: {_lbl} (level={_lvl})")


### Run: GSM8K

In [ ]:
# Block - GSM8K driver (RocketKV, this notebook's single level). Independent
# of ARC-Challenge/HellaSwag -- run this cell on its own if you only want
# GSM8K results. _lbl/_lvl come from the shared setup cell above.
print(f"================  {_lbl}  |  GSM8K  ================")
gsm8k_results = [evaluate_gsm8k_rocketkv(gsm8k_qa_pairs, _lbl, _lvl)]
gsm8k_results_df = pd.DataFrame(gsm8k_results)
display(gsm8k_results_df)

### Run: ARC-Challenge

In [ ]:
# Block - ARC-Challenge driver: scores every answer choice for each
# question via the shared score_mc_question_rocketkv (Helper Functions),
# then reports character-length normalized MC accuracy -- matches
# large_sample_implementations' evaluate_arc_kvquant/h2o methodology exactly
# (teacher-forced per-choice likelihood, not generation). Aggregation mirrors
# GSM8K: perplexity = MEAN of per-question perplexities (from each question's
# gold choice), TTFT/TBT/latency = means over questions, peak_memory_mb = max
# over questions, average_memory_mb = mean over questions.


def evaluate_arc_rocketkv(items, method_label, level_value):
    clear_memory()
    correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values = [], [], [], [], []
    per_item_records = []

    N_PREVIEW_ITEMS = 5
    for idx, item in enumerate(tqdm(items, desc=f"ARC-Challenge | {method_label}")):
        prompt = f"Question: {item['question']}\nAnswer:"
        choice_texts = [text for _, text in item["choices"]]
        gold_index = next(i for i, (label, _) in enumerate(item["choices"]) if label == item["gold_label"])

        result = score_mc_question_rocketkv(prompt, choice_texts, gold_index, level_value)

        correct += result["normalized_correct"]
        total += 1

        predicted_label = item["choices"][result["normalized_prediction"]][0]
        if idx < N_PREVIEW_ITEMS:
            print(f"\n--- ARC-Challenge | {method_label} | item {idx} preview ---")
            print(f"Question:   {item['question']}")
            print(f"Choices:    {item['choices']}")
            print(f"Gold label: {item['gold_label']} | Predicted: {predicted_label} | Correct: {bool(result['normalized_correct'])}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])
        ppl_values.append(result["perplexity"])

        choices_block = "\n".join(f"{label}. {text}" for label, text in item["choices"])
        prompt_and_choices = f"{item['question']}\n{choices_block}"

        per_item_records.append({
            "item_index": idx,
            "prompt": prompt_and_choices,
            "gold_label": item["gold_label"],
            "predicted_label": predicted_label,
            "correct": result["normalized_correct"],
            "correct_raw": result["raw_correct"],
            "perplexity": result["perplexity"],
            "ttft_sec": result["ttft_sec"],
            "tbt_sec": result["tbt_sec"],
            "total_latency_sec": result["total_latency_sec"],
            "peak_memory_mb": result["peak_memory_bytes"] / 1024**2,
        })

    accuracy = correct / max(total, 1)
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)
    _per_prompt_path = f"/content/drive/MyDrive/KVQuant_v3_Results/{method_label}_arc_challenge_per_prompt.csv"
    pd.DataFrame(per_item_records).to_csv(_per_prompt_path, index=False)
    print(f"Saved {len(per_item_records)} per-item ARC-Challenge rows to {_per_prompt_path}")

    return {
        "dataset": "ARC-Challenge",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_mb": max(peak_mem_values) / 1024**2 if peak_mem_values else 0.0,
        "average_memory_mb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**2) if peak_mem_values else 0.0,
    }


arc_results = [evaluate_arc_rocketkv(arc_items, _lbl, _lvl)]
arc_results_df = pd.DataFrame(arc_results)
display(arc_results_df)


### Run: HellaSwag

In [ ]:
# Block - HellaSwag driver: scores every answer choice for each example
# via the shared score_mc_question_rocketkv (Helper Functions) -- the
# same machinery ARC-Challenge uses, since both are likelihood-scored MC
# datasets. Aggregation is identical to ARC-Challenge: normalized accuracy,
# perplexity = MEAN of per-question perplexities (from each example's gold
# ending), TTFT/TBT/latency = means over examples, peak_memory_mb = max over
# examples, average_memory_mb = mean over examples.


def evaluate_hellaswag_rocketkv(items, method_label, level_value):
    clear_memory()
    correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values = [], [], [], [], []
    per_item_records = []

    N_PREVIEW_ITEMS = 5
    for idx, item in enumerate(tqdm(items, desc=f"HellaSwag | {method_label}")):
        result = score_mc_question_rocketkv(item["prompt"], item["choices"], item["gold_index"], level_value)

        correct += result["normalized_correct"]
        total += 1

        if idx < N_PREVIEW_ITEMS:
            print(f"\n--- HellaSwag | {method_label} | item {idx} preview ---")
            print(f"Prompt:     {item['prompt']}")
            print(f"Choices:    {item['choices']}")
            print(f"Gold index: {item['gold_index']} | Predicted: {result['normalized_prediction']} | Correct: {bool(result['normalized_correct'])}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])
        ppl_values.append(result["perplexity"])

        letters = "ABCDEFGHIJ"[:len(item["choices"])]
        choices_block = "\n".join(f"{letter}. {choice}" for letter, choice in zip(letters, item["choices"]))
        prompt_and_choices = f"{item['prompt']}\n{choices_block}"

        per_item_records.append({
            "item_index": idx,
            "prompt": prompt_and_choices,
            "gold_index": item["gold_index"],
            "predicted_index": result["normalized_prediction"],
            "correct": result["normalized_correct"],
            "correct_raw": result["raw_correct"],
            "perplexity": result["perplexity"],
            "ttft_sec": result["ttft_sec"],
            "tbt_sec": result["tbt_sec"],
            "total_latency_sec": result["total_latency_sec"],
            "peak_memory_mb": result["peak_memory_bytes"] / 1024**2,
        })

    accuracy = correct / max(total, 1)
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)
    _per_prompt_path = f"/content/drive/MyDrive/KVQuant_v3_Results/{method_label}_hellaswag_per_prompt.csv"
    pd.DataFrame(per_item_records).to_csv(_per_prompt_path, index=False)
    print(f"Saved {len(per_item_records)} per-item HellaSwag rows to {_per_prompt_path}")

    return {
        "dataset": "HellaSwag",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_mb": max(peak_mem_values) / 1024**2 if peak_mem_values else 0.0,
        "average_memory_mb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**2) if peak_mem_values else 0.0,
    }


hellaswag_results = [evaluate_hellaswag_rocketkv(hellaswag_items, _lbl, _lvl)]
hellaswag_results_df = pd.DataFrame(hellaswag_results)
display(hellaswag_results_df)


### Run: RULER


In [ ]:
# Block - RULER driver (RocketKV). Independent of GSM8K/ARC-Challenge/
# HellaSwag -- run this cell on its own if you only want RULER results;
# _lbl/_lvl come from the shared setup cell above "### Run: GSM8K". Mirrors
# generate_gsm8k_rocketkv/evaluate_gsm8k_rocketkv's structure exactly
# (rocketkv_prefill/rocketkv_step, budget resolved from THIS item's own
# prompt length, same TTFT/TBT/latency definitions), scoring a single
# generated answer per item rather than choices. Perplexity is the mean NLL
# of the model's own generated tokens (no "####"-style span to isolate
# here, so it covers the whole short answer span, unlike GSM8K's
# marker-scoped perplexity).
#
# CSV note: full_prompt is NOT written per row here (unlike GSM8K) -- a
# RULER prompt is up to ~16K tokens of synthetic haystack text, and writing
# that out 2048 times would bloat the CSV for no analytical benefit;
# length_bucket identifies which of the three context lengths each row is.


@torch.no_grad()
def generate_ruler_rocketkv(prompt, level_value, mode=None):
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_len = enc["input_ids"].shape[1]
    t = resolve_token_budget(prompt_len, level_value, mode or BUDGET_MODE)

    sync_if_cuda(); gen_start = time.perf_counter()
    last_logits, pkv, sizes, hsa, peak_bytes = rocketkv_prefill(enc["input_ids"], t)
    next_id = int(last_logits.argmax(dim=-1)[0].item())
    sync_if_cuda(); ttft_sec = time.perf_counter() - gen_start

    page_size = sizes[1]
    generated_ids = []
    generated_logits = []
    n_forward_tokens = 1

    for step in range(RULER_MAX_NEW_TOKENS):
        if next_id == tokenizer.eos_token_id:
            break
        generated_ids.append(next_id)
        generated_logits.append(last_logits.detach())
        if "\n" in tokenizer.decode([next_id]):
            break

        if step == RULER_MAX_NEW_TOKENS - 1:
            break

        token_tensor = torch.tensor([[next_id]], dtype=torch.long, device=device)
        last_logits, pkv = rocketkv_step(token_tensor, prompt_len + step, pkv, sizes, hsa)
        next_id = int(last_logits.argmax(dim=-1)[0].item())
        n_forward_tokens += 1
        peak_bytes = max(peak_bytes, rocketkv_cache_bytes(pkv, page_size, hsa))

    sync_if_cuda(); gen_end = time.perf_counter()
    total_latency_sec = gen_end - gen_start

    if generated_ids:
        nll_sum = 0.0
        for tok_id, logits in zip(generated_ids, generated_logits):
            lp = torch.log_softmax(logits[0].float(), dim=-1)
            nll_sum += -lp[tok_id].item()
        perplexity = math.exp(min(nll_sum / len(generated_ids), 50.0))
    else:
        perplexity = None

    gen_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    n_generated = n_forward_tokens
    if len(generated_ids) == 0:
        ttft_sec = total_latency_sec
    tbt_sec = (total_latency_sec - ttft_sec) / max(n_generated - 1, 1)

    return {
        "prefill_tokens": prompt_len, "generated_tokens": len(generated_ids), "gen_text": gen_text,
        "ttft_sec": ttft_sec, "tbt_sec": tbt_sec, "total_latency_sec": total_latency_sec,
        "total_tokens": prompt_len + n_generated, "peak_memory_bytes": peak_bytes, "perplexity": perplexity,
    }


def evaluate_ruler_rocketkv(items, method_label, level_value):
    clear_memory()
    correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values = [], [], [], [], []
    per_question_records = []

    N_PREVIEW_QUESTIONS = 5
    for q_idx, item in enumerate(tqdm(items, desc=f"RULER | {method_label}")):
        result = generate_ruler_rocketkv(item["prompt"], level_value)
        is_correct = item["gold_value"] in result["gen_text"]
        correct += int(is_correct)
        total += 1

        if q_idx < N_PREVIEW_QUESTIONS:
            print(f"\n--- RULER | {method_label} | question {q_idx} ({item['target_tokens']} tokens) preview ---")
            print(f"Gold value: {item['gold_value']}")
            print(f"Generated:  {result['gen_text'].strip()!r}")
            print(f"Correct: {is_correct}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])
        if result["perplexity"] is not None:
            ppl_values.append(result["perplexity"])

        per_question_records.append({
            "question_index": q_idx,
            "length_bucket": item["target_tokens"],
            "full_prompt": item["prompt"],
            "prefill_tokens": result["prefill_tokens"],
            "generated_output": result["gen_text"],
            "generated_tokens": result["generated_tokens"],
            "gold_answer": item["gold_value"],
            "correct": int(is_correct),
            "perplexity": result["perplexity"],
            "ttft_sec": result["ttft_sec"],
            "tbt_sec": result["tbt_sec"],
            "total_latency_sec": result["total_latency_sec"],
            "peak_memory_mb": result["peak_memory_bytes"] / 1024**2,
        })

    accuracy = correct / max(total, 1)
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)
    _per_prompt_path = f"/content/drive/MyDrive/KVQuant_v3_Results/{method_label}_ruler_per_prompt.csv"
    pd.DataFrame(per_question_records).to_csv(_per_prompt_path, index=False)
    print(f"Saved {len(per_question_records)} per-question RULER rows to {_per_prompt_path}")

    return {
        "dataset": "RULER",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_mb": max(peak_mem_values) / 1024**2 if peak_mem_values else 0.0,
        "average_memory_mb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**2) if peak_mem_values else 0.0,
    }


ruler_results = [evaluate_ruler_rocketkv(ruler_items, _lbl, _lvl)]
ruler_results_df = pd.DataFrame(ruler_results)
display(ruler_results_df)


### Run: SQuAD1.1


In [ ]:
# Block - SQuAD1.1 driver (RocketKV). Independent of GSM8K/ARC-Challenge/
# HellaSwag/RULER -- run this cell on its own if you only want SQuAD1.1
# results; _lbl/_lvl come from the shared setup cell above "### Run: GSM8K".
# Mirrors generate_ruler_rocketkv/evaluate_ruler_rocketkv's structure
# exactly (rocketkv_prefill/rocketkv_step, budget resolved from THIS
# item's own prompt length), scoring a short generated answer per question
# against SQuAD's official Exact Match / F1 metrics (best-of however many
# gold reference answers a question has). SQuAD passages are short, so
# (like GSM8K/ARC-Challenge/HellaSwag) budgets derived at aggressive
# compression ratios will often land in RocketKV's no-op regime -- see the
# notebook's top-of-file caveat.


@torch.no_grad()
def generate_squad_rocketkv(prompt, level_value, mode=None):
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_len = enc["input_ids"].shape[1]
    t = resolve_token_budget(prompt_len, level_value, mode or BUDGET_MODE)

    sync_if_cuda(); gen_start = time.perf_counter()
    last_logits, pkv, sizes, hsa, peak_bytes = rocketkv_prefill(enc["input_ids"], t)
    next_id = int(last_logits.argmax(dim=-1)[0].item())
    sync_if_cuda(); ttft_sec = time.perf_counter() - gen_start

    page_size = sizes[1]
    generated_ids = []
    generated_logits = []
    n_forward_tokens = 1

    for step in range(SQUAD_MAX_NEW_TOKENS):
        if next_id == tokenizer.eos_token_id:
            break
        generated_ids.append(next_id)
        generated_logits.append(last_logits.detach())
        if "\n" in tokenizer.decode([next_id]):
            break

        if step == SQUAD_MAX_NEW_TOKENS - 1:
            break

        token_tensor = torch.tensor([[next_id]], dtype=torch.long, device=device)
        last_logits, pkv = rocketkv_step(token_tensor, prompt_len + step, pkv, sizes, hsa)
        next_id = int(last_logits.argmax(dim=-1)[0].item())
        n_forward_tokens += 1
        peak_bytes = max(peak_bytes, rocketkv_cache_bytes(pkv, page_size, hsa))

    sync_if_cuda(); gen_end = time.perf_counter()
    total_latency_sec = gen_end - gen_start

    if generated_ids:
        nll_sum = 0.0
        for tok_id, logits in zip(generated_ids, generated_logits):
            lp = torch.log_softmax(logits[0].float(), dim=-1)
            nll_sum += -lp[tok_id].item()
        perplexity = math.exp(min(nll_sum / len(generated_ids), 50.0))
    else:
        perplexity = None

    gen_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    n_generated = n_forward_tokens
    if len(generated_ids) == 0:
        ttft_sec = total_latency_sec
    tbt_sec = (total_latency_sec - ttft_sec) / max(n_generated - 1, 1)

    return {
        "prefill_tokens": prompt_len, "generated_tokens": len(generated_ids), "gen_text": gen_text,
        "ttft_sec": ttft_sec, "tbt_sec": tbt_sec, "total_latency_sec": total_latency_sec,
        "total_tokens": prompt_len + n_generated, "peak_memory_bytes": peak_bytes, "perplexity": perplexity,
    }


def evaluate_squad_rocketkv(items, method_label, level_value):
    clear_memory()
    correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values, f1_values = [], [], [], [], [], []
    per_question_records = []

    N_PREVIEW_QUESTIONS = 5
    for q_idx, item in enumerate(tqdm(items, desc=f"SQuAD1.1 | {method_label}")):
        prompt = format_squad_prompt(item)
        result = generate_squad_rocketkv(prompt, level_value)
        prediction = result["gen_text"].strip()
        em = squad_exact_match(prediction, item["gold_answers"])
        f1 = squad_f1(prediction, item["gold_answers"])
        correct += em
        total += 1
        f1_values.append(f1)

        if q_idx < N_PREVIEW_QUESTIONS:
            print(f"\n--- SQuAD1.1 | {method_label} | question {q_idx} preview ---")
            print(f"Question:     {item['question']}")
            print(f"Gold answers: {item['gold_answers']}")
            print(f"Generated:    {prediction!r}")
            print(f"EM: {em} | F1: {f1:.3f}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])
        if result["perplexity"] is not None:
            ppl_values.append(result["perplexity"])

        per_question_records.append({
            "question_index": q_idx,
            "full_prompt": prompt,
            "prefill_tokens": result["prefill_tokens"],
            "generated_output": result["gen_text"],
            "generated_tokens": result["generated_tokens"],
            "gold_answer": " / ".join(item["gold_answers"]),
            "predicted_answer": prediction,
            "correct": em,
            "f1": f1,
            "perplexity": result["perplexity"],
            "ttft_sec": result["ttft_sec"],
            "tbt_sec": result["tbt_sec"],
            "total_latency_sec": result["total_latency_sec"],
            "peak_memory_mb": result["peak_memory_bytes"] / 1024**2,
        })

    accuracy = correct / max(total, 1)
    avg_f1 = sum(f1_values) / len(f1_values) if f1_values else float("nan")
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)
    _per_prompt_path = f"/content/drive/MyDrive/KVQuant_v3_Results/{method_label}_squad_per_prompt.csv"
    pd.DataFrame(per_question_records).to_csv(_per_prompt_path, index=False)
    print(f"Saved {len(per_question_records)} per-question SQuAD1.1 rows to {_per_prompt_path}")

    return {
        "dataset": "SQuAD1.1",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy,
        "avg_f1": avg_f1,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_mb": max(peak_mem_values) / 1024**2 if peak_mem_values else 0.0,
        "average_memory_mb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**2) if peak_mem_values else 0.0,
    }


squad_results = [evaluate_squad_rocketkv(squad_items, _lbl, _lvl)]
squad_results_df = pd.DataFrame(squad_results)
display(squad_results_df)


## Save Results

In [ ]:
# Block - Save combined RocketKV results to CSV. Robust to partial runs:
# only concatenates whichever of gsm8k_results_df / arc_results_df /
# hellaswag_results_df / ruler_results_df actually exist in this session, so
# you can run just a subset of datasets' cells without this cell crashing on
# a NameError for a dataframe you never created -- matching the
# Baseline/H2O/KVQuant notebooks' convention exactly. RULER in particular
# may not always finish (16K-token eager attention is memory-heavy), so this
# matters more here than it did before RULER existed.

_result_df_names = ["gsm8k_results_df", "arc_results_df", "hellaswag_results_df", "ruler_results_df", "squad_results_df"]
_available_dfs = []
for _name in _result_df_names:
    if _name in globals():
        _available_dfs.append(globals()[_name])
    else:
        print(f"Note: {_name} not found in this session -- skipping (its dataset's cells were not run).")

results_df = pd.concat(_available_dfs, ignore_index=True)
results_df = results_df[[
    "dataset", "method", "perplexity", "accuracy",
    "ttft_sec", "tbt_sec", "avg_total_latency_sec",
    "peak_memory_mb", "average_memory_mb",
]]
display(results_df)

os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)
_suffix = "" if _HSA_OK else "_stage1only"
_path = f"/content/drive/MyDrive/KVQuant_v3_Results/rocketkv{_suffix}_results.csv"
results_df.to_csv(_path, index=False)
print(f"Saved to {_path}")
print("\nREMINDER: GSM8K/ARC-Challenge/HellaSwag/SQuAD1.1 are short-context; RocketKV is a "
      "long-context method. Treat those four as a functional/regime check, not a comparison "
      "table -- RULER (up to 16K tokens) is the one dataset here actually inside RocketKV's "
      "designed regime, though still well short of the paper's 16K-109K range. Add LongBench "
      "for even longer-context coverage.")


In [ ]:
from google.colab import runtime
runtime.unassign()